# HIV transcription foci: slice-wise background subtraction, projection, and auto-thresholding

Goal: segment bright HIV transcription foci against dimmer diffuse cytoplasmic signal.

Pipeline (one processing cell, one QC cell):
1. Load a multi-dimensional TIF and keep the selected channel
2. Max-intensity Z-project the **raw** channel (kept only for the QC plot)
3. Rolling-ball background subtraction, **slice by slice** (2D ball, no Z coupling)
4. Max-intensity Z-project the background-subtracted stack
5. Median filter the projection (denoising, so the threshold sees a cleaner histogram)
6. Convert 16-bit -> 8-bit (this is the input to thresholding)
7. Try all auto-threshold methods
8. Apply the chosen threshold and show a 4-panel QC plot

Background is subtracted per slice *before* projecting because rolling ball and max are
both non-linear: subtracting from the projection has to fit one ball to the combined haze
of every plane, whereas per-slice subtraction removes each plane's own out-of-focus
background and pulls every slice to the same zero baseline before they compete in the max.

Thresholding uses the ImageJ / Fiji **Auto_Threshold** methods (all 17 of them) via
`AutoThresholder.py` in this folder, rather than scikit-image's threshold functions,
so the levels match what Fiji would pick. That file is a vendored copy of
[lamdao/vipy](https://github.com/lamdao/vipy/blob/master/AutoThresholder.py) (GPL-3.0,
a port of G. Landini's plugin); its docstring lists the Python 3 and bug fixes applied.

Every method takes a 256-bin histogram of an 8-bit image and returns a level `t`;
the mask is `image > t`, which is Fiji's "dark background" convention.

This notebook runs in a `uv`-managed environment scoped to this folder (`pyproject.toml` / `.venv` here). Run it with `uv run jupyter lab`, or select the "Python (zhiling-marko-hiv)" kernel if opening in an IDE.


## Load modules


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tifffile

from skimage.filters import median
from skimage.morphology import disk
from skimage.restoration import rolling_ball
from skimage.exposure import rescale_intensity
from skimage.util import img_as_ubyte

import AutoThresholder  # ImageJ / Fiji Auto_Threshold methods, vendored in this folder


## Parameters


In [ ]:
IMAGE_PATH = "/Volumes/Jeff-exFAT/Marko_HIV/110723/MARF1/Dox/MARF1_Dox_48hpi_2023_07_31__18_28_19.czi_-_Image_2__04_Series_04.tif"

CHANNEL_AXIS = 1   # 0-based in the order of most likely (Z, C, Y, X); check with the optional shape cell below
CHANNEL_INDEX = 0  # 0-based: which channel to keep
Z_AXIS = 0         # O-based: Z axis of the stack *after* the channel axis has been removed

ROLLING_BALL_RADIUS = 10  # pixels; must be larger than the foci, smaller than cell-scale variation
MEDIAN_RADIUS = 2         # pixels; a disk(2) footprint is 5x5, so foci < ~3 px across get eroded

# Any ImageJ Auto_Threshold method: DefaultIsoData, Huang, IJIsoData, Intermodes,
# IsoData, Li, MaxEntropy, Mean, MinError, Minimum, Moments, Otsu, Percentile,
# RenyiEntropy, Shanbhag, Triangle, Yen
THRESHOLD_METHOD = "MaxEntropy"

# Display only: percentile range used to auto-contrast the QC panels
CONTRAST_PERCENTILES = (0.5, 99.9)


## Helpers

Thin wrappers around `AutoThresholder.py` (build the 256-bin histogram the ImageJ methods
expect, look a method up by name) plus a display-only auto-contrast helper for the plots.


In [ ]:
# the 17 Auto_Threshold methods, in the plugin's own order
METHOD_NAMES = sorted(
    (name for name in vars(AutoThresholder.Methods) if not name.startswith("__")),
    key=lambda name: getattr(AutoThresholder.Methods, name),
)


def histogram_8bit(img):
    """256-bin histogram of an 8-bit image, as the ImageJ methods expect."""
    return np.bincount(np.asarray(img, dtype=np.uint8).ravel(), minlength=256)


def auto_threshold(img, method, hist=None):
    """ImageJ auto-threshold level for an 8-bit image; -1 if the method finds none."""
    if hist is None:
        hist = histogram_8bit(img)
    fn = AutoThresholder.Fx[getattr(AutoThresholder.Methods, method)]
    with np.errstate(divide="ignore", invalid="ignore"):
        # Triangle and IJIsoData edit the histogram in place, so hand them a copy
        return int(fn(hist.copy()))


def contrast_limits(img, percentiles=CONTRAST_PERCENTILES):
    """(vmin, vmax) for display, from robust percentiles; does not alter the data."""
    lo, hi = np.percentile(img, percentiles)
    return float(lo), float(hi if hi > lo else lo + 1)


print(f"{len(METHOD_NAMES)} methods: " + ", ".join(METHOD_NAMES))


## Optional: inspect the image shape

Run this once on a new dataset to confirm `CHANNEL_AXIS`, `CHANNEL_INDEX` and `Z_AXIS`.
Not needed for the processing cell below.


In [ ]:
raw_stack = tifffile.imread(IMAGE_PATH)
print(f"full stack shape: {raw_stack.shape}, dtype: {raw_stack.dtype}")
channel_stack = np.take(raw_stack, indices=CHANNEL_INDEX, axis=CHANNEL_AXIS)
print(f"selected channel shape (expect Z, Y, X): {channel_stack.shape}")


## Process: slice-wise background subtraction, projection, denoising, all thresholds

Everything up to the threshold survey in one cell. Outputs kept for the QC cell:

- `raw_projection` -- max Z-projection of the untouched channel (QC reference only)
- `input_8bit` -- the 8-bit image that is actually thresholded
- `threshold_levels` -- level picked by each ImageJ method (-1 if none found)


In [ ]:
# --- 1. Open the image and keep only the selected channel -------------------------------
raw_stack = tifffile.imread(IMAGE_PATH)
channel_stack = np.take(raw_stack, indices=CHANNEL_INDEX, axis=CHANNEL_AXIS)  # (Z, Y, X)

# --- 2. Max Z-projection of the raw channel, kept only for the QC plot ------------------
raw_projection = np.max(channel_stack, axis=Z_AXIS)

# --- 3. Rolling-ball background subtraction, slice by slice ----------------------------
# A 2D ball per slice, not a 3D kernel: each plane's own out-of-focus haze is removed and
# every slice is pulled to a zero baseline before the slices compete in the max projection.
subtracted_slices = []
for z_slice in np.moveaxis(channel_stack, Z_AXIS, 0):
    background = rolling_ball(z_slice, radius=ROLLING_BALL_RADIUS)
    subtracted = np.clip(z_slice.astype(np.float64) - background, 0, None)
    subtracted_slices.append(subtracted.astype(channel_stack.dtype))
subtracted_stack = np.stack(subtracted_slices, axis=0)

# --- 4. Max Z-projection of the background-subtracted stack ----------------------------
subtracted_projection = np.max(subtracted_stack, axis=0)

# --- 5. Median filter the projection ---------------------------------------------------
# Denoising only (edge-preserving, not edge-enhancing): removes single-pixel spikes so the
# histogram the threshold methods see is cleaner and the mask is less speckled.
denoised_projection = median(subtracted_projection, disk(MEDIAN_RADIUS))

# --- 6. Convert 16-bit -> 8-bit; this is the input to thresholding ---------------------
input_8bit = img_as_ubyte(rescale_intensity(denoised_projection, out_range=(0, 1)))

# --- 7. Try all ImageJ auto-threshold methods -----------------------------------------
input_hist = histogram_8bit(input_8bit)
threshold_levels = {}
for name in METHOD_NAMES:
    try:
        threshold_levels[name] = auto_threshold(input_8bit, name, input_hist)
    except Exception as exc:  # a method can fail outright on a degenerate histogram
        threshold_levels[name] = -1
        print(f"{name}: failed ({type(exc).__name__}: {exc})")

print(f"channel stack {channel_stack.shape} -> projection {input_8bit.shape}, "
      f"8-bit min/max {input_8bit.min()}/{input_8bit.max()}")
for name, level in threshold_levels.items():
    if level < 0:
        print(f"{name:<15} ->  no threshold found")
    else:
        print(f"{name:<15} -> {level:4d}   foreground {100 * (input_8bit > level).mean():6.2f}%")

# --- 8. Plot the 8-bit input and every usable method's mask ---------------------------
usable_methods = [(name, level) for name, level in threshold_levels.items() if level >= 0]

ncols = 6
nrows = -(-(len(usable_methods) + 1) // ncols)  # +1 for the input image
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.6 * nrows))
axes = np.atleast_1d(axes).ravel()

vmin, vmax = contrast_limits(input_8bit)
axes[0].imshow(input_8bit, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("8-bit input", fontsize=9)
for ax, (name, level) in zip(axes[1:], usable_methods):
    ax.imshow(input_8bit > level, cmap="gray")
    ax.set_title(f"{name} ({level})", fontsize=9)
for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Apply the chosen threshold and QC plot

Four panels, all auto-contrasted for display: raw max projection, 8-bit input, mask, overlay.


In [ ]:
# --- Apply the chosen threshold -------------------------------------------------------
threshold_level = threshold_levels.get(THRESHOLD_METHOD, auto_threshold(input_8bit, THRESHOLD_METHOD))
if threshold_level < 0:
    raise ValueError(f"{THRESHOLD_METHOD} found no threshold for this image")

foci_mask = input_8bit > threshold_level  # Fiji "dark background" convention
print(f"{THRESHOLD_METHOD} threshold level: {threshold_level}"
      f"   foreground {100 * foci_mask.mean():.2f}%")

# --- QC plot: raw projection | 8-bit input | mask | overlay ----------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

vmin, vmax = contrast_limits(raw_projection)
axes[0].imshow(raw_projection, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Raw channel, max Z-projection")

vmin, vmax = contrast_limits(input_8bit)
axes[1].imshow(input_8bit, cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("Background-subtracted + median, 8-bit input")

axes[2].imshow(foci_mask, cmap="gray")
axes[2].set_title(f"Mask ({THRESHOLD_METHOD} > {threshold_level})")

# orange where the mask is True, fully transparent elsewhere
orange = ListedColormap(["#ff6600"])
axes[3].imshow(input_8bit, cmap="gray", vmin=vmin, vmax=vmax)
axes[3].imshow(np.ma.masked_where(~foci_mask, foci_mask), cmap=orange, alpha=0.9, vmin=0, vmax=1)
axes[3].set_title("Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()
